# Query Answering without Pretrained Models

Notebook này xây một hệ thống trả lời query cho dữ liệu trong thư mục `data` bằng Information Retrieval cổ điển. Pipeline dùng BM25, stemming, stopword removal, n-gram và exact phrase boost từ chính project hiện tại.

**Không sử dụng pretrained model:** không dùng LLM, embedding model, transformer, sentence-transformer, word2vec/GloVe pretrained, hay API bên ngoài. Kết quả trả về là danh sách tài liệu liên quan và snippet bằng chứng từ corpus Cranfield.

## 1. Cấu hình đường dẫn

In [ ]:
from pathlib import Path
import csv
import os
import sys
import textwrap

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
CORPUS_DIR = DATA_DIR / "Cranfield"
QUERY_CSV = DATA_DIR / "public_test_queries.csv"
ANSWER_CSV = DATA_DIR / "public_test_answers.csv"
OUTPUT_DIR = ROOT / "submissions"
OUTPUT_CSV = OUTPUT_DIR / "no_pretrained_bm25_submission.csv"

# Cho phép notebook import các baseline/helper nội bộ của project.
PROJECT_IR = ROOT / "ir_baseline" / "ir_baseline"
if str(PROJECT_IR) not in sys.path:
    sys.path.insert(0, str(PROJECT_IR))

print("Project root:", ROOT)
print("Corpus dir  :", CORPUS_DIR)
print("Query file  :", QUERY_CSV)

## 2. Load dữ liệu trong `data`

In [ ]:
from utils.data_loader import load_answers, load_corpus, load_queries, save_submission

corpus = load_corpus(str(CORPUS_DIR))
queries = load_queries(str(QUERY_CSV))
answers = load_answers(str(ANSWER_CSV)) if ANSWER_CSV.exists() else {}

print(f"Loaded {len(corpus)} documents")
print(f"Loaded {len(queries)} queries")
print(f"Loaded {len(answers)} answer rows")

sample_qid = sorted(queries)[0]
print("\nSample query")
print(sample_qid, queries[sample_qid])

## 3. Khởi tạo retriever không pretrained

`EnhancedBM25Retriever` trong repo là mô hình sparse retrieval tự tính thống kê trên corpus hiện tại. Nó chỉ học các đại lượng như term frequency, document frequency, IDF và độ dài tài liệu từ dữ liệu local.

In [ ]:
from baselines.baseline4_bm25_ngram import EnhancedBM25Retriever

retriever = EnhancedBM25Retriever(
    k1=1.5,
    b=0.75,
    stem=True,
    max_ngram=3,
    bigram_weight=0.12,
    trigram_weight=0.18,
    exact_phrase_boost=0.10,
    candidate_pool=120,
)

retriever.fit(corpus)

## 4. Hàm trả lời một query

Hàm dưới đây nhận query text hoặc `query_id`, sau đó trả về top tài liệu phù hợp nhất cùng đoạn trích trong tài liệu.

In [ ]:
def compact_whitespace(text):
    return " ".join(text.split())


def make_snippet(text, query, width=360):
    """Chọn snippet quanh term query đầu tiên tìm được; fallback lấy đầu document."""
    raw = compact_whitespace(text)
    lowered = raw.lower()
    query_terms = [t for t in query.lower().replace("/", " ").split() if len(t) > 3]
    hit_at = None
    for term in query_terms:
        pos = lowered.find(term)
        if pos >= 0:
            hit_at = pos
            break
    if hit_at is None:
        hit_at = 0

    start = max(0, hit_at - width // 3)
    end = min(len(raw), start + width)
    snippet = raw[start:end]
    if start > 0:
        snippet = "... " + snippet
    if end < len(raw):
        snippet = snippet + " ..."
    return snippet


def answer_query(query_or_id, top_k=5, snippet_width=360):
    if isinstance(query_or_id, int):
        if query_or_id not in queries:
            raise KeyError(f"Không có query_id={query_or_id} trong {QUERY_CSV}")
        qid = query_or_id
        query = queries[qid]
    else:
        qid = None
        query = str(query_or_id)

    ranked = retriever.query(query, top_k=top_k)
    rows = []
    for rank, (doc_id, score) in enumerate(ranked, start=1):
        rows.append({
            "rank": rank,
            "doc_id": doc_id,
            "score": round(score, 4),
            "snippet": make_snippet(corpus[doc_id], query, width=snippet_width),
        })

    print("Query ID:", qid if qid is not None else "custom")
    print("Query   :", query)
    print("-" * 90)
    for row in rows:
        print(f"#{row['rank']} | doc_id={row['doc_id']} | score={row['score']}")
        print(textwrap.fill(row["snippet"], width=100))
        print()
    return rows

## 5. Demo trả lời query trong folder `data`

In [ ]:
# Có thể đổi query_id thành các ID trong data/public_test_queries.csv
answer_query(62, top_k=5)

In [ ]:
# Hoặc nhập query tự do, vẫn chỉ truy hồi trên corpus local.
answer_query("boundary layer flow around a cylinder", top_k=5)

## 6. Chạy toàn bộ query và đánh giá

In [ ]:
from utils.evaluate import evaluate, evaluate_per_query

TOP_K = 20
results = retriever.retrieve_all(queries, top_k=TOP_K)

if answers:
    print(f"Evaluation with top_k={TOP_K}")
    metrics = evaluate(results, answers, verbose=True)
else:
    metrics = {}
    print("Không tìm thấy answer file, bỏ qua evaluate.")

OUTPUT_DIR.mkdir(exist_ok=True)
save_submission(results, str(OUTPUT_CSV))

## 7. Xem các query còn yếu để tuning thủ công

In [ ]:
if answers:
    report = evaluate_per_query(results, answers)
    worst = sorted(report.items(), key=lambda item: item[1]["f1"])[:5]
    for qid, info in worst:
        print("=" * 90)
        print(f"query_id={qid} | f1={info['f1']:.4f} | relevant={answers[qid]}")
        print("query:", queries[qid])
        print("predicted:", info["predicted"][:10])
        print("relevant ranks:", info["relevant_ranks"])

## Ghi chú ràng buộc

- Notebook này không tải model, không gọi internet, không gọi API bên ngoài.
- Các tham số BM25/ngram là heuristic, có thể tuning bằng validation nếu có thêm nhãn.
- Vì không dùng generative pretrained model, câu trả lời được biểu diễn bằng tài liệu liên quan và snippet thay vì sinh văn bản tự do.